# MixedGAM / ConditionalSNN 統一示範（California Housing）

較大資料集的完整範例，包含：
1) group_affine 訓練與解釋，以 Latitude 分箱作群組
2) 提供可直接調整的 `MixedGAMConfig`
3) 評估使用 `root_mean_squared_error` (RMSE)
4) 與 CatBoost (僅數值特徵) 比較
5) 展示 `feature_group_map` 介面（目前簡化為所有特徵共用同一個群組欄位）


## 1. 匯入套件與設定
在專案根目錄下執行，Notebook 會把 `src` 加到路徑。


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from catboost import CatBoostRegressor

project_root = Path('..').resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.models import ConditionalSNNRegressor, MixedGAMConfig

plt.style.use('seaborn-v0_8')
np.random.seed(42)



## 2. 讀取與前處理資料（California Housing）
- 來源：`fetch_california_housing`
- 群組欄位：以 `Latitude` 分成 5 個分箱 (`lat_bin`)
- 特徵：全部連續數值
- 兩份資料：
  - `X_gam`：直接使用數值特徵，給 ConditionalSNN。
  - `X_cat`：同樣數值特徵，給 CatBoost。


In [ ]:
cal = fetch_california_housing(as_frame=True)
df = cal.frame.copy()
target_col = 'MedHouseVal'
feature_cols = cal.feature_names
df = df.dropna(subset=feature_cols + [target_col])

# 群組欄位：Latitude 分箱
lat_bin = pd.qcut(df['Latitude'], 5, labels=False).astype(int).astype(str)
groups_df = pd.DataFrame({'lat_bin': lat_bin})
X = df[feature_cols]
y = df[target_col]

X_train, X_test, y_train, y_test, g_train, g_test = train_test_split(
    X, y, groups_df, test_size=0.2, random_state=42, stratify=lat_bin
)
print(f'California housing shapes: X={X.shape}, y={y.shape}, groups={groups_df.shape}')



## 3. `MixedGAMConfig` 範例設定
可依需求修改超參數。


In [ ]:
cfg = MixedGAMConfig(
    mode='group_affine',
    scaler='standard',
    base_hidden_units=(128, 64, 32),
    base_activation='tanh',
    base_dropout=0.1,
    base_norm='layernorm',
    residual_hidden_units=(64, 32),
    residual_activation='relu',
    residual_dropout=0.2,
    residual_norm='layernorm',
    learning_rate=1e-3,
    weight_decay=5e-5,
    batch_size=512,
    n_epochs=120,
    patience=25,
    lambda_center=5e-4,
    lambda_group_l1=5e-4,
    lambda_group_l2=1e-4,
    lambda_orth=1e-4,
    lambda_affine_scale=5e-4,
    lambda_affine_shift=5e-4,
    lambda_affine_center=1e-4,
    lambda_base_target=5e-4,
    residual_contribution_l1=0.0,
    residual_contribution_l2=0.0,
    parallel_feature_mlp=False,
    verbose=True,
    random_state=42,
)
cfg



## 4. 訓練 ConditionalSNN (group_affine, RMSE 評估)


In [ ]:
feature_group_map = {name: 'lat_bin' for name in X.columns}
model = ConditionalSNNRegressor(**cfg.__dict__)
model.fit(
    X_train, y_train, groups=g_train,
    X_val=X_test, y_val=y_test, groups_val=g_test,
    progress_refresh_rate=5,
    feature_group_map=feature_group_map,
)



## 5. 評估 + 重要性
使用 `root_mean_squared_error`，並檢查 `feature_importance_` / `feature_names_in_`。


In [ ]:
preds = model.predict(X_test, groups=g_test)
rmse = root_mean_squared_error(y_test, preds)
mae = mean_absolute_error(y_test, preds)
r2 = r2_score(y_test, preds)
print(f'ConditionalSNN → RMSE={rmse:.4f}, MAE={mae:.4f}, R2={r2:.4f}')

fi = model.feature_importance()
fi_sorted = pd.Series(fi).sort_values(ascending=False)
display(fi_sorted)
print('feature_names_in_ length:', len(model.feature_names_in_))



## 6. 群組形狀函數示範
選擇一個特徵與群組，畫出 base shape 與 affine warp。


In [ ]:
feature = 'Latitude'
g = g_test['lat_bin'].unique()[0]
xs, base_shape = model.get_feature_shape(feature)
xs_g, group_shape = model.get_group_affine_shape(feature, g)
plt.figure(figsize=(6,4))
plt.plot(xs, base_shape, label='Global base', linewidth=2)
plt.plot(xs_g, group_shape, label=f'Affine for group={g}', linewidth=2)
plt.xlabel(feature)
plt.ylabel('Contribution')
plt.legend()
plt.tight_layout()
plt.show()



## 7. CatBoost 比較（數值特徵）


In [ ]:
cat_model = CatBoostRegressor(
    depth=8,
    learning_rate=0.05,
    loss_function='RMSE',
    n_estimators=800,
    random_seed=42,
    verbose=False,
)
cat_model.fit(X_train, y_train)
cat_preds = cat_model.predict(X_test)
cat_rmse = root_mean_squared_error(y_test, cat_preds)
cat_mae = mean_absolute_error(y_test, cat_preds)
cat_r2 = r2_score(y_test, cat_preds)
print(f'CatBoost → RMSE={cat_rmse:.4f}, MAE={cat_mae:.4f}, R2={cat_r2:.4f}')



## 8. 存取/載入示範
保存模型與 metadata，並重新載入檢查。


In [ ]:
artifact_dir = Path('artifacts')
artifact_dir.mkdir(exist_ok=True)
model_path = artifact_dir / 'mixed_gam_demo.pt'
meta_path = artifact_dir / 'mixed_gam_demo_meta.pkl'

model.save(model_path, meta_path)
restored = ConditionalSNNRegressor.load(model_path, meta_path)
print('Restored feature_names_in_ length:', len(restored.feature_names_in_))

